In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
import soundfile as sf
from facenet_pytorch import InceptionResnetV1

# Configuración de Seaborn
sns.set_theme(style="whitegrid")

# Rutas Base
SEEN_DATASET_ROOT = "/home/voces/datasets/lrs3/trainval_processed"
UNSEEN_DATASET_ROOT = "/home/voces/datasets/lrs3/pretrain_processed"
MODELS_ROOT = "/home/voces/datasets/face_project_models_exp"
STYLETTS_ROOT = "/home/voces/code/StyleTTS2"

# Añadir StyleTTS2 al path
sys.path.append(STYLETTS_ROOT)
try:
    from models import StyleEncoder
    from meldataset import preprocess
except ImportError:
    print("Error importando StyleTTS2. Verifica la ruta.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## 2. Definición de la Arquitectura del Modelo (Flexible)

Definimos la clase `FaceToVoiceModel` correspondiente a la arquitectura "Flexible".

In [ ]:
class FaceToVoiceModel(nn.Module):
    def __init__(self, style_encoder_checkpoint=None, freeze_visual=True, soft_tuning=False):
        super().__init__()

        # A. Visual Encoder (FaceNet)
        # Pretrained on vggface2
        self.visual_encoder = InceptionResnetV1(pretrained='vggface2')

        # Output dimension of InceptionResnetV1 is 512 by default (if classify=False)
        self.d_vis = 512

        # B. Audio Encoder (Target - Frozen)
        # We need to instantiate StyleEncoder.
        # "timbre, 64-dim". So style_dim=64.
        # Standard StyleTTS2 config usually has dim_in=64 (channels), max_conv_dim=512.
        # We assume these defaults.
        self.audio_encoder = StyleEncoder(dim_in=64, style_dim=128, max_conv_dim=512)
        self.audio_encoder.eval() # Set to eval mode

        # C. Projection Head (Adapter)
        # FaceNet outputs 512 dim. StyleTTS2 expects 128 dim.
        self.projector_base = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.projector_head = nn.Linear(1024, 128)

        # D. Auxiliary Heads
        # Gender: 128 -> 1 (Sigmoid)
        self.gender_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1) # Logits for BCEWithLogitsLoss
        )

        # Age: 128 -> 1 (Linear/MSE)
        self.age_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward_visual(self, images):
        # images: [B, 3, 160, 160]
        face_emb = self.visual_encoder(images) # [B, 512]

        # Projector
        hidden = self.projector_base(face_emb) # [B, 1024]

        # Main embedding
        proj_emb = self.projector_head(hidden) # [B, 128]

        # Aux outputs (using proj_emb to force latent space structure)
        pred_gender = self.gender_head(proj_emb)
        pred_age = self.age_head(proj_emb)

        proj_emb = F.normalize(proj_emb, p=2, dim=1)

        return proj_emb, pred_gender, pred_age

    def forward_audio(self, mels):
        # mels: [B, 80, T]
        with torch.no_grad():
            # StyleEncoder expects [B, 1, 80, T] because of Conv2d(1, ...)
            mels = mels.unsqueeze(1)

            # StyleEncoder forward returns style embedding
            # We assume it returns [B, 128]
            audio_emb = self.audio_encoder(mels)

            # Normalize
            audio_emb = F.normalize(audio_emb, p=2, dim=1)

        return audio_emb

def load_model(checkpoint_path):
    model = FaceToVoiceModel()
    # Cargar pesos ignorando errores de tamaño si hay diferencias menores en cabeceras aux
    # Pero para análisis necesitamos que coincida.
    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu')
        if 'model_state_dict' in ckpt:
            state_dict = ckpt['model_state_dict']
        else:
            state_dict = ckpt

        model.load_state_dict(state_dict, strict=False)
        print(f"Modelo cargado desde {checkpoint_path}")
    except Exception as e:
        print(f"Error cargando {checkpoint_path}: {e}")
        return None

    model.to(device)
    model.eval()
    return model

## 3. Carga del Dataset (LRS3 Pretrain Processed)

Cargamos una muestra del dataset para validación. Usaremos `metadata_train.csv` (o val) de la carpeta `pretrain_processed`.

In [ ]:
class LRS3AnalysisDataset(Dataset):
    def __init__(self, root_dir, metadata_file='metadata_train.csv', transform=None, max_samples=1000):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        # Cargar atributos
        attr_file = os.path.join(root_dir, 'face_attributes.json')
        self.attributes = {}
        if os.path.exists(attr_file):
            with open(attr_file, 'r') as f:
                self.attributes = json.load(f)

        # Cargar metadatos
        meta_path = os.path.join(root_dir, metadata_file)
        print(f"Cargando metadatos desde {meta_path}...")

        with open(meta_path, 'r') as f:
            lines = f.readlines()

        # Filtrar candidatos válidos primero
        valid_samples = []

        for line in lines:
            parts = line.strip().split('|')
            if len(parts) >= 1:
                audio_path = parts[0]

                # 1. Verificar atributos (Género y Edad)
                if audio_path not in self.attributes:
                    continue

                attrs = self.attributes[audio_path]
                gender = attrs.get('gender')
                age = attrs.get('age')

                # Filtrar si no tiene atributos válidos
                if gender is None or age is None:
                    continue

                # 2. Verificar Archivos e Identidad (Imagen correspondiente)
                base_name = os.path.splitext(audio_path)[0]
                if '/audio/' in base_name:
                    base_name_img = base_name.replace('/audio/', '/images/')
                else:
                    base_name_img = base_name

                img_path_png = base_name_img + ".png"
                img_path_jpg = base_name_img + ".jpg"

                final_img_path = None
                if os.path.exists(img_path_png):
                    final_img_path = img_path_png
                elif os.path.exists(img_path_jpg):
                    final_img_path = img_path_jpg

                if final_img_path and os.path.exists(audio_path):
                    speaker_id = parts[2] if len(parts) > 2 else "unknown"

                    # Asegurar identidad válida
                    if speaker_id == "unknown":
                        continue

                    valid_samples.append({
                        'audio_path': audio_path,
                        'img_path': final_img_path,
                        'speaker_id': speaker_id,
                        'gender': gender,
                        'age': age
                    })

        # Muestrear de los válidos
        if max_samples and len(valid_samples) > max_samples:
            np.random.seed(42)
            indices = np.random.choice(len(valid_samples), max_samples, replace=False)
            self.samples = [valid_samples[i] for i in indices]
        else:
            self.samples = valid_samples

        print(f"Dataset cargado: {len(self.samples)} muestras (de {len(lines)} líneas originales).")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Imagen
        try:
            image = Image.open(sample['img_path']).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except:
            image = torch.zeros(3, 160, 160)

        # Audio (Mel)
        try:
            wav, sr = sf.read(sample['audio_path'])
            # Preprocess StyleTTS2
            mel = preprocess(wav).squeeze(0) # [80, T]

            # Crop/Pad fijo para batching
            target_len = 200
            if mel.size(1) > target_len:
                mel = mel[:, :target_len]
            else:
                mel = F.pad(mel, (0, target_len - mel.size(1)))
        except:
            mel = torch.zeros(80, 200)

        return {
            'image': image,
            'mel': mel,
            'speaker_id': sample['speaker_id'],
            'gender': sample['gender'],
            'age': sample['age']
        }

# Transformaciones
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Instanciar Dataset (Usamos metadata_val.csv si existe, sino train)
# Ajustar nombre de archivo según lo visto en list_dir
dataset = LRS3AnalysisDataset(SEEN_DATASET_ROOT, metadata_file='metadata_train_validated.csv', transform=transform, max_samples=500)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

## 4. Extracción de Embeddings

Función para extraer embeddings visuales y de audio de todo el dataset.

In [ ]:
def extract_embeddings(model, dataloader):
    visual_embs = []
    audio_embs = []
    metadata = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extrayendo embeddings"):
            images = batch['image'].to(device)
            mels = batch['mel'].to(device)

            # Forward
            v_emb, _, _ = model.forward_visual(images)
            a_emb = model.forward_audio(mels)

            visual_embs.append(v_emb.cpu().numpy())
            audio_embs.append(a_emb.cpu().numpy())

            # Metadata
            for i in range(len(batch['speaker_id'])):
                metadata.append({
                    'speaker_id': batch['speaker_id'][i],
                    'gender': batch['gender'][i].item(),
                    'age': batch['age'][i].item()
                })

    return np.concatenate(visual_embs), np.concatenate(audio_embs), pd.DataFrame(metadata)

def calculate_metrics(v_embs, a_embs):
    # Similitud Coseno Promedio (Pares correspondientes)
    # Asumimos que v_embs[i] corresponde a a_embs[i]
    # Los vectores ya están normalizados por el modelo
    cosine_sims = (v_embs * a_embs).sum(axis=1)
    avg_sim = cosine_sims.mean()

    return avg_sim, cosine_sims

## 5. Visualización con UMAP

Proyectamos ambos espacios (Visual y Audio) en el mismo plano 2D para ver su alineación.

In [ ]:
def plot_umap(v_embs, a_embs, metadata, title="UMAP Projection", save_path=None):
    # Combinar para UMAP conjunto
    combined_embs = np.vstack([v_embs, a_embs])

    # Ajustar UMAP
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
    embedding = reducer.fit_transform(combined_embs)

    n = len(v_embs)
    v_proj = embedding[:n]
    a_proj = embedding[n:]

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 1. Por Modalidad (Cara vs Voz)
    axes[0].scatter(v_proj[:, 0], v_proj[:, 1], c='blue', alpha=0.5, label='Cara', s=10)
    axes[0].scatter(a_proj[:, 0], a_proj[:, 1], c='red', alpha=0.5, label='Voz', s=10)
    # Dibujar líneas conectando pares (solo una muestra aleatoria para no saturar)
    indices = np.random.choice(n, min(50, n), replace=False)
    for i in indices:
        axes[0].plot([v_proj[i, 0], a_proj[i, 0]], [v_proj[i, 1], a_proj[i, 1]], 'k-', alpha=0.1)
    axes[0].set_title(f"{title} - Alineación modalidad")
    axes[0].legend()

    # 2. Por Género (Usando embeddings visuales)
    scatter = axes[1].scatter(v_proj[:, 0], v_proj[:, 1], c=metadata['gender'], cmap='coolwarm', alpha=0.7, s=15)
    axes[1].set_title("Espacio visual por género")
    plt.colorbar(scatter, ax=axes[1])

    # 3. Por Edad
    scatter = axes[2].scatter(v_proj[:, 0], v_proj[:, 1], c=metadata['age'], cmap='viridis', alpha=0.7, s=15)
    axes[2].set_title("Espacio visual por edad")
    plt.colorbar(scatter, ax=axes[2])

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
        print(f"Gráfica guardada en: {save_path}")

    plt.show()

## 6. Análisis Evolutivo: Modelo Flexible

Analizamos checkpoints específicos para ver la evolución.

In [ ]:
# Configuración de modelos a evaluar

models_config = [
    {
        "name": "Exp1 Identity (EN)",
        "dir_name": "checkpoints_exp1_identity_en"
    }
]

# Checkpoints a evaluar
checkpoints = [
    "face_adapter_ep5.pth",
    "face_adapter_ep25.pth",
    "face_adapter_ep50.pth",
    "face_adapter_ep100.pth",
    "face_adapter_ep150.pth",
    "best_model_top1.pth"
]

# Directorio de salida para imágenes
output_dir = "analysis/plots_flexible"
os.makedirs(output_dir, exist_ok=True)

results_all = []

# Helper: generar una única imagen por modelo con una fila por checkpoint
def plot_umap_grid(ckpt_results, model_name, save_dir):
    if not ckpt_results:
        return

    n_rows = len(ckpt_results)
    cols = 3  # Modalidad, género, edad (como plot_umap)
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 6, n_rows * 5))
    if n_rows == 1:
        axes = np.array([axes])

    for i, res in enumerate(ckpt_results):
        v_embs = res['v_embs']
        a_embs = res['a_embs']
        meta = res['meta']
        ckpt = res['ckpt']
        avg_sim = res['avg_sim']

        combined_embs = np.vstack([v_embs, a_embs])
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
        embedding = reducer.fit_transform(combined_embs)

        n = len(v_embs)
        v_proj = embedding[:n]
        a_proj = embedding[n:]

        # 1. Modalidad
        ax0 = axes[i, 0] if n_rows > 1 else axes[0]
        ax0.scatter(v_proj[:, 0], v_proj[:, 1], c='blue', alpha=0.5, label='Cara', s=10)
        ax0.scatter(a_proj[:, 0], a_proj[:, 1], c='red', alpha=0.5, label='Voz', s=10)
        idxs = np.random.choice(n, min(50, n), replace=False)
        for j in idxs:
            ax0.plot([v_proj[j, 0], a_proj[j, 0]], [v_proj[j, 1], a_proj[j, 1]], 'k-', alpha=0.1)
        ax0.set_title(f"{ckpt} | sim={avg_sim:.4f}")
        ax0.legend()

        # 2. Género
        ax1 = axes[i, 1] if n_rows > 1 else axes[1]
        sc1 = ax1.scatter(v_proj[:, 0], v_proj[:, 1], c=meta['gender'], cmap='coolwarm', alpha=0.7, s=15)
        ax1.set_title("Género (visual)")
        plt.colorbar(sc1, ax=ax1)

        # 3. Edad
        ax2 = axes[i, 2] if n_rows > 1 else axes[2]
        sc2 = ax2.scatter(v_proj[:, 0], v_proj[:, 1], c=meta['age'], cmap='viridis', alpha=0.7, s=15)
        ax2.set_title("Edad (visual)")
        plt.colorbar(sc2, ax=ax2)

    fig.suptitle(f"Evolución UMAP por checkpoint | {model_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])

    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "").lower()
    save_path = os.path.join(save_dir, f"{safe_name}_umap_grid.png")
    plt.savefig(save_path, dpi=300)
    print(f"Gráfica consolidada guardada en: {save_path}")
    plt.show()

for model_cfg in models_config:
    model_name = model_cfg["name"]
    dir_name = model_cfg["dir_name"]
    model_dir = os.path.join(MODELS_ROOT, dir_name)

    print(f"\n=== Evaluando Modelo: {model_name} ===")

    if not os.path.exists(model_dir):
        print(f"Directorio no encontrado: {model_dir}")
        continue

    ckpt_results = []

    for ckpt_name in checkpoints:
        ckpt_path = os.path.join(model_dir, ckpt_name)
        if not os.path.exists(ckpt_path):
            print(f"Saltando {ckpt_name} (no encontrado)")
            continue

        print(f"--- Procesando {ckpt_name} ---")
        model = load_model(ckpt_path)
        if model:
            v_embs, a_embs, meta = extract_embeddings(model, dataloader)
            avg_sim, _ = calculate_metrics(v_embs, a_embs)
            print(f"Similitud Coseno Promedio: {avg_sim:.4f}")

            ckpt_results.append({
                'ckpt': ckpt_name,
                'v_embs': v_embs,
                'a_embs': a_embs,
                'meta': meta,
                'avg_sim': avg_sim
            })

            results_all.append({
                'model': model_name,
                'epoch': ckpt_name,
                'sim': avg_sim
            })

    # Generar imagen consolidada por modelo
    plot_umap_grid(ckpt_results, model_name, output_dir)

In [ ]:
# --- Configuración para Inferencia y Generación de HTML ---

# Añadir directorio padre al path para importar inference_utils
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

try:
    from inference_utils import StyleTTS2Inference
    import soundfile as sf
except ImportError:
    print("Error: No se pudo importar inference_utils. Verifica que estás en el directorio correcto.")

# Configuración de Modelos Base (StyleTTS2)
base_tts_config = {
    "Hybrid (EN)": {
        "config": "/home/voces/code/StyleTTS2/Models/LibriTTS/config.yml",
        "ckpt": "/home/voces/code/StyleTTS2/Models/LibriTTS/epochs_2nd_00020.pth",
        "text": "This is a voice generated from a face image using the English model.",
        "lang": "en-us"
    },
    "Hybrid (ES)": {
        "config": "/home/voces/code/StyleTTS2/Models/Fonos/config_ft_es-ca_resume_v2.yml",
        "ckpt": "/home/voces/code/StyleTTS2/Models/Fonos/epoch_2nd_00026.pth",
        "text": "Esta es una voz generada a partir de una imagen de rostro usando el modelo en español.",
        "lang": "es"
    }
}

# Parámetros de Inferencia
INFERENCE_PARAMS = {
    "alpha": 0.5,
    "beta": 1.0,
    "diffusion_steps": 25,
    "embedding_scale": 1.0,
    "speed": 1.0
}

# Directorio de salida para audios
AUDIO_OUTPUT_DIR = "analysis/generated_samples_flexible"
os.makedirs(AUDIO_OUTPUT_DIR, exist_ok=True)

In [ ]:

checkpoints = [
    "face_adapter_ep25.pth",
    "face_adapter_ep50.pth",
    "face_adapter_ep150.pth",
    "best_model_top1.pth"
]

# --- Selección de Muestras Diversas (Vistos y No Vistos) ---

# Cargar dataset de validación (No Vistos) para comparar
if 'val_dataset' not in globals():
    print("Cargando dataset de validación (Speakers No Vistos)...")
    val_dataset = LRS3AnalysisDataset(UNSEEN_DATASET_ROOT, metadata_file='metadata_val.csv', transform=transform, max_samples=500)
else:
    print("Dataset de validación ya cargado.")

def select_diverse_samples(dataset, n_per_gender=3, dataset_type='seen'):
    """Selecciona n hombres y n mujeres con edades variadas, asegurando speakers únicos."""
    males = []
    females = []

    # Agrupar muestras por speaker para elegir una representativa (ej: la primera)
    seen_speakers = set()
    unique_samples = []

    # Primero recolectamos candidatos únicos
    for i in range(len(dataset)):
        sample = dataset.samples[i]
        if sample['speaker_id'] not in seen_speakers:
            s_copy = sample.copy()
            s_copy['idx'] = i
            s_copy['dataset_type'] = dataset_type # Etiquetar tipo de dataset
            unique_samples.append(s_copy)
            seen_speakers.add(sample['speaker_id'])

    # Clasificar por género
    for sample in unique_samples:
        g = sample['gender']
        is_male = False
        if isinstance(g, str):
            if g.lower() == 'male':
                is_male = True
        elif isinstance(g, (int, float)):
            if g == 1:
                is_male = True

        if is_male:
            males.append(sample)
        else:
            females.append(sample)

    # Ordenar por edad
    males.sort(key=lambda x: x['age'])
    females.sort(key=lambda x: x['age'])

    def pick_spread(lst, n):
        if len(lst) < n:
            return lst
        indices = np.linspace(0, len(lst) - 1, n, dtype=int)
        return [lst[i] for i in indices]

    selected_males = pick_spread(males, n_per_gender)
    selected_females = pick_spread(females, n_per_gender)

    return selected_males + selected_females

# Seleccionar muestras de ambos conjuntos
samples_seen = select_diverse_samples(dataset, n_per_gender=4, dataset_type='seen')
samples_unseen = select_diverse_samples(val_dataset, n_per_gender=4, dataset_type='unseen')

selected_samples = samples_seen + samples_unseen

print(f"Seleccionadas {len(selected_samples)} muestras totales.")

# --- Bucle de Generación de Audio ---

import torchaudio

# Definir configuración de modelos para inferencia
inference_models_config = [
    {
        "name": "Exp1 Identity (EN)",
        "dir_name": "checkpoints_exp1_identity_en",
        "tts_base_key": "Hybrid (EN)"
    },
    {
        "name": "Exp1 Identity (ES)",
        "dir_name": "checkpoints_exp1_identity",
        "tts_base_key": "Hybrid (ES)"
    }
]

def generate_audio_samples(models_config, checkpoints, selected_samples):
    generation_results = {} # Estructura: Model -> Checkpoint -> [Samples]

    for model_cfg in models_config:
        model_name = model_cfg["name"]
        dir_name = model_cfg["dir_name"]
        tts_key = model_cfg.get("tts_base_key", model_name) # Fallback

        model_dir = os.path.join(MODELS_ROOT, dir_name)

        # Configuración Base TTS
        if tts_key not in base_tts_config:
            print(f"Advertencia: No hay configuración base TTS para {tts_key}")
            continue

        tts_cfg = base_tts_config[tts_key]

        print(f"\n=== Generando para Modelo: {model_name} ===")
        print(f"Usando Face Adapter de: {dir_name}")
        print(f"Usando StyleTTS2 Base ({tts_cfg['lang']}): {tts_cfg['ckpt']}")

        # Cargar StyleTTS2 (Inferencia)
        try:
            inference_engine = StyleTTS2Inference(
                model_checkpoint=tts_cfg['ckpt'],
                config_path=tts_cfg['config'],
                device=device,
                language=tts_cfg.get('lang', 'en')
            )
        except Exception as e:
            print(f"Error cargando StyleTTS2 para {model_name}: {e}")
            continue

        generation_results[model_name] = {}

        for ckpt_name in checkpoints:
            ckpt_path = os.path.join(model_dir, ckpt_name)
            if not os.path.exists(ckpt_path):
                continue

            print(f"--- Checkpoint: {ckpt_name} ---")

            # Cargar Face Adapter
            face_model = load_model(ckpt_path)
            if not face_model:
                continue

            generation_results[model_name][ckpt_name] = []

            for sample in tqdm(selected_samples, desc="Generando audios"):
                # Preparar imagen
                try:
                    # Cargar imagen original desde path
                    img_path = sample['img_path']
                    image = Image.open(img_path).convert('RGB')
                    image_tensor = transform(image).unsqueeze(0).to(device)

                    # Obtener embedding
                    with torch.no_grad():
                        out_visual = face_model.forward_visual(image_tensor)
                        # Manejar tupla (emb, gender, age)
                        if isinstance(out_visual, tuple):
                            style_emb = out_visual[0]
                        else:
                            style_emb = out_visual

                    # Inferencia
                    wav = inference_engine.inference(
                        text=tts_cfg['text'],
                        style_emb=style_emb,
                        ref_s=None, # Sin referencia de audio, solo cara
                        alpha=INFERENCE_PARAMS['alpha'],
                        beta=INFERENCE_PARAMS['beta'],
                        diffusion_steps=INFERENCE_PARAMS['diffusion_steps'],
                        embedding_scale=INFERENCE_PARAMS['embedding_scale'],
                        speed=INFERENCE_PARAMS['speed']
                    )

                    # Guardar Audio
                    safe_model = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("->", "to").lower()
                    safe_ckpt = ckpt_name.replace(".pth", "")
                    safe_id = sample['speaker_id']
                    safe_type = sample['dataset_type']
                    filename = f"{safe_model}_{safe_ckpt}_{safe_type}_{safe_id}_{sample['gender']}_{sample['age']}.wav"
                    filepath = os.path.join(AUDIO_OUTPUT_DIR, filename)

                    sf.write(filepath, wav, 24000)

                    # Guardar info para HTML
                    generation_results[model_name][ckpt_name].append({
                        'sample': sample,
                        'audio_path': filepath,
                        'filename': filename
                    })

                except Exception as e:
                    print(f"Error generando muestra {sample['speaker_id']}: {e}")

            # Liberar memoria del face model
            del face_model
            torch.cuda.empty_cache()

        # Liberar engine
        del inference_engine
        torch.cuda.empty_cache()

    return generation_results

# Ejecutar generación
gen_results = generate_audio_samples(inference_models_config, checkpoints, selected_samples)

In [ ]:
# --- Organización de Assets y Limpieza ---

import shutil
import json
import os

def finalize_report_assets(results, analysis_root="analysis"):
    """
    Copia imágenes y audios originales a carpetas locales, limpia audios generados no usados
    y regenera el HTML con rutas relativas.
    """
    images_dir = os.path.join(analysis_root, "report_images")
    orig_audio_dir = os.path.join(analysis_root, "original_audios")

    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(orig_audio_dir, exist_ok=True)

    used_audios = set()

    print("Iniciando organización de assets...")

    js_data = {}

    # 1. Copiar imágenes, audios originales y recolectar audios generados usados
    for model_name, checkpoints_data in results.items():
        js_data[model_name] = {}
        for ckpt_name, samples in checkpoints_data.items():
            js_data[model_name][ckpt_name] = []
            for item in samples:
                # --- Audio Generado ---
                audio_abs = os.path.abspath(item['audio_path'])
                used_audios.add(audio_abs)
                rel_audio = os.path.relpath(item['audio_path'], start=analysis_root)

                # --- Imagen ---
                src_img = item['sample']['img_path']
                img_name = f"{item['sample']['speaker_id']}_{os.path.basename(src_img)}"
                dst_img = os.path.join(images_dir, img_name)

                if not os.path.exists(dst_img):
                    try:
                        shutil.copy2(src_img, dst_img)
                    except Exception as e:
                        print(f"Error copiando imagen {src_img}: {e}")
                rel_img = f"report_images/{img_name}"

                # --- Audio Original ---
                src_orig = item['sample']['audio_path']
                orig_ext = os.path.splitext(src_orig)[1]
                if not orig_ext: orig_ext = ".wav"
                orig_name = f"{item['sample']['speaker_id']}_original{orig_ext}"
                dst_orig = os.path.join(orig_audio_dir, orig_name)

                if not os.path.exists(dst_orig):
                    try:
                        shutil.copy2(src_orig, dst_orig)
                    except Exception as e:
                        print(f"Error copiando audio original {src_orig}: {e}")
                rel_orig_audio = f"original_audios/{orig_name}"

                # --- Metadatos ---
                g_val = item['sample']['gender']
                gender_str = "Hombre" if (g_val == 1 or str(g_val).lower() == 'male') else "Mujer"

                dataset_type = item['sample'].get('dataset_type', 'unknown')

                js_data[model_name][ckpt_name].append({
                    'img': rel_img,
                    'audio': rel_audio,
                    'original_audio': rel_orig_audio,
                    'gender': gender_str,
                    'age': item['sample']['age'],
                    'id': item['sample']['speaker_id'],
                    'type': dataset_type
                })

    # 2. Limpiar audios generados no usados
    # Nota: Limpiamos solo en la carpeta flexible
    audio_dir = os.path.join(analysis_root, "generated_samples_flexible")
    deleted_count = 0
    if os.path.exists(audio_dir):
        for f in os.listdir(audio_dir):
            if not f.endswith('.wav'):
                continue
            f_path = os.path.join(audio_dir, f)
            f_abs = os.path.abspath(f_path)
            if f_abs not in used_audios:
                try:
                    os.remove(f_abs)
                    deleted_count += 1
                except Exception as e:
                    pass
    print(f"Se eliminaron {deleted_count} archivos de audio generados no utilizados.")

    # 3. Generar HTML con Tailwind y JS
    json_data = json.dumps(js_data)

    html_content = f"""
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Reporte Evolución Face-to-Voice (Flexible)</title>
        <script src="https://cdn.tailwindcss.com"></script>
    </head>
    <body class="bg-gray-100 text-gray-800 font-sans p-6">
        <div class="max-w-7xl mx-auto">
            <header class="mb-8 text-center">
                <h1 class="text-3xl font-bold text-blue-700 mb-2">Reporte de Evolución Face-to-Voice (Flexible)</h1>
                <p class="text-gray-600 max-w-2xl mx-auto">
                    Análisis del modelo Flexible Identity.
                </p>
            </header>

            <div class="bg-white rounded-lg shadow-md p-6 mb-8">
                <div class="flex flex-col md:flex-row gap-4 justify-center items-center">
                    <div class="w-full md:w-1/3">
                        <label for="modelSelect" class="block text-sm font-medium text-gray-700 mb-1">Modelo / Idioma</label>
                        <select id="modelSelect" class="w-full p-2 border border-gray-300 rounded-md focus:ring-blue-500 focus:border-blue-500">
                        </select>
                    </div>
                    <div class="w-full md:w-1/3">
                        <label for="ckptSelect" class="block text-sm font-medium text-gray-700 mb-1">Checkpoint</label>
                        <select id="ckptSelect" class="w-full p-2 border border-gray-300 rounded-md focus:ring-blue-500 focus:border-blue-500">
                        </select>
                    </div>
                    <div class="w-full md:w-1/3">
                        <label for="typeSelect" class="block text-sm font-medium text-gray-700 mb-1">Tipo de Speaker</label>
                        <select id="typeSelect" class="w-full p-2 border border-gray-300 rounded-md focus:ring-blue-500 focus:border-blue-500">
                            <option value="all">Todos</option>
                            <option value="seen">Vistos (Train)</option>
                            <option value="unseen" selected>No Vistos (Val/Test)</option>
                        </select>
                    </div>
                </div>
            </div>

            <div id="samplesGrid" class="grid grid-cols-1 sm:grid-cols-2 md:grid-cols-3 lg:grid-cols-4 gap-6">
            </div>

            <div id="emptyState" class="hidden text-center py-12 text-gray-500">
                No hay datos para la selección actual.
            </div>
        </div>

        <script>
            const data = {json_data};
            const modelSelect = document.getElementById('modelSelect');
            const ckptSelect = document.getElementById('ckptSelect');
            const typeSelect = document.getElementById('typeSelect');
            const grid = document.getElementById('samplesGrid');
            const emptyState = document.getElementById('emptyState');

            const models = Object.keys(data);
            models.forEach(m => {{
                const opt = document.createElement('option');
                opt.value = m;
                opt.textContent = m;
                modelSelect.appendChild(opt);
            }});

            function updateCheckpoints() {{
                const model = modelSelect.value;
                ckptSelect.innerHTML = '';

                if (data[model]) {{
                    const ckpts = Object.keys(data[model]);
                    ckpts.sort((a, b) => {{
                        const numA = parseInt(a.replace(/[^0-9]/g, '')) || 0;
                        const numB = parseInt(b.replace(/[^0-9]/g, '')) || 0;
                        return numA - numB;
                    }});

                    ckpts.forEach(c => {{
                        const opt = document.createElement('option');
                        opt.value = c;
                        opt.textContent = c;
                        ckptSelect.appendChild(opt);
                    }});
                }}
                updateGrid();
            }}

            function updateGrid() {{
                const model = modelSelect.value;
                const ckpt = ckptSelect.value;
                const typeFilter = typeSelect.value;

                grid.innerHTML = '';
                let hasData = false;

                if (data[model] && data[model][ckpt]) {{
                    const samples = data[model][ckpt];

                    samples.forEach(s => {{
                        if (typeFilter !== 'all' && s.type !== typeFilter) return;

                        hasData = true;
                        const card = document.createElement('div');
                        card.className = 'bg-white rounded-lg shadow overflow-hidden flex flex-col items-center p-4 hover:shadow-lg transition-shadow relative';

                        const badgeColor = s.type === 'seen' ? 'bg-green-100 text-green-800' : 'bg-purple-100 text-purple-800';
                        const badgeText = s.type === 'seen' ? 'Visto' : 'No Visto';

                        card.innerHTML = `
                            <span class="absolute top-2 right-2 px-2 py-1 text-xs font-semibold rounded-full ${{badgeColor}}">
                                ${{badgeText}}
                            </span>
                            <div class="w-24 h-24 mb-3 relative mt-2">
                                <img src="${{s.img}}" alt="Face" class="w-full h-full object-cover rounded-full border-2 border-blue-100">
                            </div>
                            <div class="text-center mb-3">
                                <div class="font-bold text-gray-800">${{s.gender}}</div>
                                <div class="text-sm text-gray-600">Edad: ${{s.age}}</div>
                                <div class="text-xs text-gray-400 mt-1">ID: ${{s.id}}</div>
                            </div>
                            <div class="w-full mt-auto space-y-3">
                                <div>
                                    <div class="text-xs text-blue-600 font-bold mb-1 uppercase tracking-wide">Generado</div>
                                    <audio controls src="${{s.audio}}" class="w-full h-8"></audio>
                                </div>
                                <div>
                                    <div class="text-xs text-gray-500 font-bold mb-1 uppercase tracking-wide">Original</div>
                                    <audio controls src="${{s.original_audio}}" class="w-full h-8 opacity-75"></audio>
                                </div>
                            </div>
                        `;
                        grid.appendChild(card);
                    }});
                }}

                if (hasData) {{
                    emptyState.classList.add('hidden');
                }} else {{
                    emptyState.classList.remove('hidden');
                }}
            }}

            modelSelect.addEventListener('change', updateCheckpoints);
            ckptSelect.addEventListener('change', updateGrid);
            typeSelect.addEventListener('change', updateGrid);

            if (models.length > 0) {{
                updateCheckpoints();
            }}
        </script>
    </body>
    </html>
    """

    output_file = os.path.join(analysis_root, "evolution_report_flexible.html")
    with open(output_file, "w") as f:
        f.write(html_content)

    print(f"Reporte HTML final generado en: {output_file}")

if 'gen_results' in globals():
    finalize_report_assets(gen_results)
else:
    print("Variable 'gen_results' no encontrada. Ejecuta la celda de generación primero.")